In [ ]:
# Glyphs_from_Texts.ipynb Requirements
#
# Inputs:
#   UTF-8 text files
#   unicode_to_notofonts_map.pkl
#   noto_font_table.pkl
#
# Output:
#   glyphs_from_texts.pkl
#
# Purpose:
#   Identify the unique characters required by each text,
#   determine which Noto fonts can render them,
#   allow the user to select rendering fonts,
#   and create a database of unique character × font combinations
#   for Glyph_Complexity_Calc.ipynb.
#
# The script does NOT:
#   - calculate complexity
#   - save images

# This keeps the glyph database limited to glyphs that are actually needed and it can be continually updated

In [5]:
import os
import pickle
import unicodedata
from collections import Counter

import pandas as pd

from google.colab import drive, files

In [62]:
# choose input/output locations
# "drive"    = Google Drive
# "computer" = upload/download using your computer

input_source = "drive"
output_source = "drive"

base_dir = (
    "/content/drive/MyDrive/"
    "Character Complexity"
)

font_map_filename = (
    "unicode_to_notofonts_map.pkl"
)

font_table_filename = (
    "noto_font_table.pkl"
)

output_filename = (
    "glyphs_from_texts.pkl"
)


In [15]:
# Load text files
#
# For both input sources, text_files contains
# the full path to each text file.

if input_source == "drive":

    text_dir = os.path.join(
        base_dir,
        "udhr_text_files"
    )

    if not os.path.isdir(text_dir):

        raise FileNotFoundError(
            f"Text directory not found:\n{text_dir}"
        )

    text_files = sorted(
        os.path.join(text_dir, filename)
        for filename in os.listdir(text_dir)
        if filename.endswith(".txt")
    )


elif input_source == "computer":

    print(
        "Upload the UTF-8 text files to analyze."
    )

    uploaded_texts = files.upload()

    if not uploaded_texts:

        raise ValueError(
            "No text files were uploaded."
        )

    # Uploaded files are already available in /content

    text_files = sorted(
        os.path.join("/content", filename)
        for filename in uploaded_texts
    )


else:

    raise ValueError(
        "input_source must be 'drive' or 'computer'."
    )


# Show all text files

print()
print("Text files found:")
print()

for filepath in text_files:

    print(
        f"  {os.path.basename(filepath)}"
    )

print()
print(
    f"Total text files: {len(text_files):,}"
)


Text files found:

  chn.txt
  eng.txt
  hnd.txt
  jpn.txt
  por.txt
  quy.txt
  rus.txt
  spn.txt

Total text files: 8


In [19]:
# Verify and summarize text files

texts = {}

verification_results = []


for filepath in text_files:

    filename = os.path.basename(filepath)

    language = os.path.splitext(
        filename
    )[0]


    # Read as UTF-8

    try:

        with open(
            filepath,
            "r",
            encoding="utf-8"
        ) as f:

            text = f.read()

        utf8_readable = True

    except UnicodeDecodeError as e:

        verification_results.append({
            "file": filename,
            "language": language,
            "UTF8": False,
            "NFC": None,
            "characters": None,
            "unique_characters": None,
            "status": f"UTF-8 ERROR: {e}"
        })

        continue


    # Check NFC

    is_nfc = unicodedata.is_normalized(
        "NFC",
        text
    )


    # Normalize for downstream analysis

    text = unicodedata.normalize(
        "NFC",
        text
    )


    # Reject empty files

    if not text.strip():

        verification_results.append({
            "file": filename,
            "language": language,
            "UTF8": True,
            "NFC": is_nfc,
            "characters": len(text),
            "unique_characters": 0,
            "status": "ERROR: empty or whitespace only"
        })

        continue


    # Character inventory

    counts = Counter(text)

    non_whitespace = {
        char: count
        for char, count in counts.items()
        if not char.isspace()
    }


    # Store normalized text

    texts[language] = text


    # Summary statistics

    letters = sum(
        count
        for char, count in non_whitespace.items()
        if unicodedata.category(char).startswith("L")
    )

    numbers = sum(
        count
        for char, count in non_whitespace.items()
        if unicodedata.category(char).startswith("N")
    )

    punctuation = sum(
        count
        for char, count in non_whitespace.items()
        if unicodedata.category(char).startswith("P")
    )

    symbols = sum(
        count
        for char, count in non_whitespace.items()
        if unicodedata.category(char).startswith("S")
    )


    verification_results.append({
        "file": filename,
        "language": language,
        "UTF8": utf8_readable,
        "NFC": is_nfc,
        "characters": len(text),
        "unique_characters": len(non_whitespace),
        "letters": letters,
        "numbers": numbers,
        "punctuation": punctuation,
        "symbols": symbols,
        "status": "Contains text"
    })


# Verification table

verification_table = pd.DataFrame(
    verification_results
)


print()
print("TEXT VERIFICATION")
print()

display(
    verification_table
)


TEXT VERIFICATION



,file,language,UTF8,NFC,characters,unique_characters,letters,numbers,punctuation,symbols,status
0,chn.txt,chn,True,True,5921,538,2679,75,234,0,Contains text
1,eng.txt,eng,True,True,11046,56,8673,82,197,0,Contains text
2,hnd.txt,hnd,True,True,49,24,38,0,2,0,Contains text
3,jpn.txt,jpn,True,True,4757,505,3759,92,304,0,Contains text
4,por.txt,por,True,True,11679,63,9191,83,227,30,Contains text
5,quy.txt,quy,True,True,13402,68,11381,87,243,0,Contains text
6,rus.txt,rus,True,True,12206,66,9926,92,250,0,Contains text
7,spn.txt,spn,True,True,12403,60,9779,92,230,0,Contains text


In [20]:
# Load font table: noto_font_table.pkl

if input_source == "drive":

    drive.mount("/content/drive")

    font_map_path = os.path.join(
        base_dir,
        font_map_filename
    )

    font_table_path = os.path.join(
        base_dir,
        font_table_filename
    )

elif input_source == "computer":

    print("Upload the Unicode → font map:")
    uploaded_map = files.upload()

    if not uploaded_map:
        raise ValueError(
            "No font map was uploaded."
        )

    font_map_path = next(
        iter(uploaded_map)
    )


    print()
    print("Upload the Noto font table:")
    uploaded_table = files.upload()

    if not uploaded_table:
        raise ValueError(
            "No font table was uploaded."
        )

    font_table_path = next(
        iter(uploaded_table)
    )

else:

    raise ValueError(
        "input_source must be 'drive' or 'computer'."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# Load Unicode to fonts map: unicode_to_notofonts_map.pkl

with open(
    font_map_path,
    "rb"
) as f:

    unicode_to_fonts = pickle.load(f)


font_table = pd.read_pickle(
    font_table_path
)


print(
    f"Unicode codepoints in font map: "
    f"{len(unicode_to_fonts):,}"
)

print(
    f"Fonts in font table: "
    f"{len(font_table):,}"
)


Unicode codepoints in font map: 79,029
Fonts in font table: 2,460
First key: 0
Key type: <class 'int'>
Map is keyed by integer Unicode codepoints.


In [37]:
# Build character inventory for each text

character_inventories = {}


print()
print("CHARACTER INVENTORIES")


for language, text in texts.items():

    counts = Counter(
        char
        for char in text
        if not char.isspace()
    )


    inventory = pd.DataFrame(
        [
            {
                "character": char,
                "codepoint": ord(char),
                "unicode": f"U+{ord(char):04X}",
                "unicode_name": unicodedata.name(
                    char,
                    "UNKNOWN"
                ),
                "category": unicodedata.category(char),
                "count": count
            }
            for char, count in counts.items()
        ]
    )


    inventory = (
        inventory
        .sort_values(
            "count",
            ascending=False
        )
        .reset_index(drop=True)
    )


    character_inventories[language] = inventory


    # Display summary

    print()
    print(f"{language}")

    print(
        f"Unique characters: "
        f"{len(inventory):,}"
    )


    # Columns to display

    display_columns = [
        "character",
        "codepoint",
        "unicode",
        "count"
    ]


    # Frequency of characters: Max and min 10

    if len(inventory) <= 20:

        preview = inventory[
            display_columns
        ]

    else:

        first_10 = inventory[
            display_columns
        ].head(10)

        last_10 = inventory[
            display_columns
        ].tail(10)

        preview = pd.concat(
            [
                first_10,
                last_10
            ]
        )


    display(preview)


CHARACTER INVENTORIES

chn
Unique characters: 538


,character,codepoint,unicode,count
0,的,30340,U+7684,149
1,人,20154,U+4EBA,104
2,和,21644,U+548C,80
3,",",44,U+002C,79
4,有,26377,U+6709,70
5,权,26435,U+6743,69
6,。,12290,U+3002,62
7,国,22269,U+56FD,42
8,、,12289,U+3001,41
9,利,21033,U+5229,35



eng
Unique characters: 56


,character,codepoint,unicode,count
0,e,101,U+0065,1045
1,t,116,U+0074,795
2,o,111,U+006F,706
3,n,110,U+006E,698
4,i,105,U+0069,696
5,a,97,U+0061,670
6,r,114,U+0072,607
7,s,115,U+0073,459
8,h,104,U+0068,444
9,l,108,U+006C,398



hnd
Unique characters: 24


...

,character,codepoint,unicode,count
0,a,97,U+0061,2235
1,n,110,U+006E,1124
2,i,105,U+0069,888
3,k,107,U+006B,737
4,h,104,U+0068,590
5,c,99,U+0063,582
6,p,112,U+0070,522
7,u,117,U+0075,522
8,q,113,U+0071,489
9,m,109,U+006D,420



rus
Unique characters: 66


,character,codepoint,unicode,count
0,о,1086,U+043E,1070
1,е,1077,U+0435,869
2,и,1080,U+0438,857
3,а,1072,U+0430,754
4,н,1085,U+043D,658
5,т,1090,U+0442,581
6,в,1074,U+0432,566
7,с,1089,U+0441,523
8,р,1088,U+0440,443
9,л,1083,U+043B,368



spn
Unique characters: 60


,character,codepoint,unicode,count
0,e,101,U+0065,1291
1,a,97,U+0061,1047
2,o,111,U+006F,827
3,i,105,U+0069,733
4,n,110,U+006E,703
5,s,115,U+0073,674
6,r,114,U+0072,666
7,d,100,U+0064,598
8,l,108,U+006C,536
9,c,99,U+0063,513


In [49]:
# Font lookup
# Edit this to search for any font name if one is preferred

font_search = "Noto Sans Regular"

font_lookup = font_table[
    font_table["font_name"]
    .str.contains(
        font_search,
        case=False,
        na=False
    )
].copy()

display(
    font_lookup[
        [
            "font_id",
            "font_name",
            "family_name",
            "font_path",
            "font_fileindex"
        ]
    ]
)

,font_id,font_name,family_name,font_path,font_fileindex
2028,2028,Noto Sans Regular,Noto Sans,/usr/share/fonts/truetype/noto/NotoSans-Regula...,0


In [50]:
# Check if preferred font covers every character in every text

selected_font_id = font_lookup.iloc[0]["font_id"]

coverage_results = []

for language, inventory in character_inventories.items():

    supported = inventory["codepoint"].apply(
        lambda cp:
        selected_font_id
        in unicode_to_fonts
        .get(cp, {})
        .get("font_ids", [])
    )

    total_characters = len(inventory)
    supported_characters = supported.sum()
    missing_characters = total_characters - supported_characters

    coverage_results.append({
        "language": language,
        "unique_characters": total_characters,
        "renderable": supported_characters,
        "missing": missing_characters,
        "full_coverage": missing_characters == 0
    })


coverage_table = pd.DataFrame(
    coverage_results
)

display(coverage_table)

,language,unique_characters,renderable,missing,full_coverage
0,chn,538,17,521,False
1,eng,56,56,0,True
2,hnd,24,24,0,True
3,jpn,505,11,494,False
4,por,63,63,0,True
5,quy,68,68,0,True
6,rus,66,66,0,True
7,spn,60,60,0,True


In [47]:
# If preferred font does not cover every text:
# # Find fonts that can render every character in each text

universal_fonts_by_language = {}

print()
print("UNIVERSAL FONT COVERAGE BY TEXT")

for language, inventory in character_inventories.items():

    font_sets = []

    for codepoint in inventory["codepoint"]:

        font_ids = set(
            unicode_to_fonts
            .get(codepoint, {})
            .get("font_ids", [])
        )

        font_sets.append(font_ids)

    if font_sets:

        universal_font_ids = set.intersection(
            *font_sets
        )

    else:

        universal_font_ids = set()

    universal_fonts_by_language[language] = (
        universal_font_ids
    )

    print(
        f"{language}: "
        f"{len(universal_font_ids):,} fonts"
    )


# Find fonts that can render every character
# across ALL texts


print()
print("UNIVERSAL FONT COVERAGE ACROSS ALL TEXTS")

nonempty_sets = [
    font_ids
    for font_ids in universal_fonts_by_language.values()
    if font_ids
]

if nonempty_sets:

    common_fonts_across_all_texts = set.intersection(
        *nonempty_sets
    )

else:

    common_fonts_across_all_texts = set()


print(
    f"Fonts covering every character in every text: "
    f"{len(common_fonts_across_all_texts):,}"
)


if common_fonts_across_all_texts:

    print()
    print("font_id    font_name    family_name")

    for font_id in sorted(
        common_fonts_across_all_texts
    ):

        match = font_table[
            font_table["font_id"] == font_id
        ]

        if not match.empty:

            row = match.iloc[0]

            print(
                f"{font_id:<10} "
                f"{row['font_name']:<30} "
                f"{row['family_name']}"
            )


UNIVERSAL FONT COVERAGE BY TEXT
chn: 80 fonts
eng: 406 fonts
hnd: 405 fonts
jpn: 80 fonts
por: 405 fonts
quy: 405 fonts
rus: 405 fonts
spn: 405 fonts

UNIVERSAL FONT COVERAGE ACROSS ALL TEXTS
Fonts covering every character in every text: 80

font_id    font_name    family_name
5          Noto Sans CJK JP Medium        Noto Sans CJK JP Medium
6          Noto Sans CJK KR Medium        Noto Sans CJK KR Medium
7          Noto Sans CJK SC Medium        Noto Sans CJK SC Medium
8          Noto Sans CJK TC Medium        Noto Sans CJK TC Medium
9          Noto Sans CJK HK Medium        Noto Sans CJK HK Medium
64         Noto Sans CJK JP Bold          Noto Sans CJK JP
65         Noto Sans CJK KR Bold          Noto Sans CJK KR
66         Noto Sans CJK SC Bold          Noto Sans CJK SC
67         Noto Sans CJK TC Bold          Noto Sans CJK TC
68         Noto Sans CJK HK Bold          Noto Sans CJK HK
69         Noto Sans Mono CJK JP Bold     Noto Sans Mono CJK JP
70         Noto Sans Mono CJK KR

In [60]:
# Render each text in chosen font - can be run multiple times
# incrementally creates the input for Glyph Complexity analysis.

Rendering_font = 2028

# Change Rendering_font and rerun this cell to add another font
# Existing character × font combinations are retained

# Default chosen for test: Noto Sans CJK SC — font_id 2142
# Noto Sans → sans-serif, matching the general visual character of ordinary Noto Sans.
# CJK → explicitly designed as a unified CJK family.
# SC → Simplified Chinese regional glyph forms.


print()
print("ADDING GLYPHS FOR RENDERING FONT")


# Find selected font

font_match = font_table[
    font_table["font_id"] == Rendering_font
]

if font_match.empty:

    raise ValueError(
        f"Rendering_font {Rendering_font} "
        "was not found in font_table."
    )

selected_font = font_match.iloc[0]


print(
    f"Rendering font: {selected_font['font_name']}"
)

print(
    f"font_id: {Rendering_font}"
)


# Build records for characters this font can render

new_glyphs = []


for language, inventory in character_inventories.items():

    for _, row in inventory.iterrows():

        codepoint = row["codepoint"]

        supported = (
            Rendering_font
            in unicode_to_fonts
            .get(codepoint, {})
            .get("font_ids", [])
        )

        if not supported:
            continue

        new_glyphs.append({

            "language": language,
            "character": row["character"],
            "codepoint": codepoint,
            "unicode": row["unicode"],
            "unicode_name": row["unicode_name"],
            "category": row["category"],
            "count": row["count"],

            "font_id": Rendering_font,
            "font_name": selected_font["font_name"],
            "font_path": selected_font["font_path"],
            "font_fileindex": selected_font["font_fileindex"]

        })


new_glyphs = pd.DataFrame(
    new_glyphs
)


# Initialize the cumulative database if this is the first rendering

if "unique_glyphs" not in globals():

    unique_glyphs = pd.DataFrame(
        columns=[
            "language",
            "character",
            "codepoint",
            "unicode",
            "unicode_name",
            "category",
            "count",
            "font_id",
            "font_name",
            "font_path",
            "font_fileindex"
        ]
    )


# Add new records

unique_glyphs = pd.concat(
    [
        unique_glyphs,
        new_glyphs
    ],
    ignore_index=True
)


# Remove duplicate character × font combinations

unique_glyphs = (
    unique_glyphs
    .drop_duplicates(
        subset=[
            "codepoint",
            "font_id"
        ]
    )
    .reset_index(drop=True)
)


print()
print(
    f"Characters renderable by this font: "
    f"{new_glyphs['codepoint'].nunique():,}"
)

print(
    f"Total unique glyph × font combinations: "
    f"{len(unique_glyphs):,}"
)


# Fonts currently included

fonts_included = (
    unique_glyphs[
        [
            "font_id",
            "font_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("font_id")
    .reset_index(drop=True)
)

print()
print(
    f"Fonts included: "
    f"{len(fonts_included):,}"
)

for _, row in fonts_included.iterrows():

    print(
        f"  {row['font_id']}: "
        f"{row['font_name']}"
    )


# Preview

preview_columns = [
    "character",
    "codepoint",
    "unicode",
    "font_id",
    "font_name"
]

print()
print("Glyph database preview")

if len(unique_glyphs) <= 30:

    display(
        unique_glyphs[preview_columns]
    )

else:

    display(
        pd.concat([
            unique_glyphs[preview_columns].head(10),
            unique_glyphs[preview_columns].tail(10)
        ])
    )


ADDING GLYPHS FOR RENDERING FONT
Rendering font: Noto Sans Regular
font_id: 2028

Characters renderable by this font: 131
Total unique glyph × font combinations: 1,032

Fonts included: 2
  2028: Noto Sans Regular
  2142: Noto Sans CJK SC

Glyph database preview


,character,codepoint,unicode,font_id,font_name
0,的,30340,U+7684,2142,Noto Sans CJK SC
1,人,20154,U+4EBA,2142,Noto Sans CJK SC
2,和,21644,U+548C,2142,Noto Sans CJK SC
3,",",44,U+002C,2142,Noto Sans CJK SC
4,有,26377,U+6709,2142,Noto Sans CJK SC
5,权,26435,U+6743,2142,Noto Sans CJK SC
6,。,12290,U+3002,2142,Noto Sans CJK SC
7,国,22269,U+56FD,2142,Noto Sans CJK SC
8,、,12289,U+3001,2142,Noto Sans CJK SC
9,利,21033,U+5229,2142,Noto Sans CJK SC


In [63]:
# Save database as Glyphs_from_Texts.pkl
#
# This database is the input for Glyph_Complexity_Calc.ipynb.
#
# The Glyphs_from_Texts database is built incrementally by:
      # uploading text files
      # choosing a Rendering_font
      # running the rendering cell
      # rerunning the save cell


print()
print("Saving Glyphs from Texts Database")


# Verify that glyph data exists

if "unique_glyphs" not in globals():

    raise ValueError(
        "unique_glyphs does not exist. "
        "Run the rendering cell first."
    )


if unique_glyphs.empty:

    raise ValueError(
        "unique_glyphs is empty. "
        "No glyphs are available to save."
    )


# Save to Google Drive

if output_source == "drive":

    output_path = os.path.join(
        base_dir,
        output_filename
    )

    unique_glyphs.to_pickle(
        output_path
    )

    print()
    print(
        f"Saved to Google Drive:"
    )

    print(
        output_path
    )


# Download to computer

elif output_source == "computer":

    output_path = os.path.join(
        "/content",
        output_filename
    )

    unique_glyphs.to_pickle(
        output_path
    )

    print()
    print(
        f"Saved locally:"
    )

    print(
        output_path
    )

    files.download(
        output_path
    )


else:

    raise ValueError(
        "output_source must be 'drive' or 'computer'."
    )


# Summary

print()
print(
    f"Glyph × font records: "
    f"{len(unique_glyphs):,}"
)

print(
    f"Unique characters: "
    f"{unique_glyphs['codepoint'].nunique():,}"
)

print(
    f"Fonts included: "
    f"{unique_glyphs['font_id'].nunique():,}"
)

print()
print(
    f"Output file: "
    f"{output_filename}"
)


Saving Glyphs from Texts Database

Saved to Google Drive:
/content/drive/MyDrive/Character Complexity/glyphs_from_texts.pkl

Glyph × font records: 1,032
Unique characters: 901
Fonts included: 2

Output file: glyphs_from_texts.pkl
